# Module 6: cropping systems, simulating crop rotations

The sixth day of the course covers cropping systems: simulating crop rotations, and creating the experiment
file for a rotation (sequence) analysis. The course's own material for that day is not available here, so this
module is built from DSSAT's sequence examples in dssat-csm-data, the engine's source code and Bowen et al.
(1998) on simulating cropping sequences with DSSAT.

1. The anatomy of a sequence experiment file (`.SQX`).
2. How the engine runs a sequence (mode `Q`).
3. Two examples: soybean or dry bean followed by fallow at Gainesville (`UFGA7803.SQX`), and soybean, chickpea and
   fallow at Hyderabad (`ITHY7501.SQX`).
4. Creating a sequence experiment: the Hyderabad rotation without the chickpea, written in code, compared with the
   full rotation over 25 years.

In [1]:
import sys, time, re
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))      # docs/notebooks
import learnlib as L

M = "module6"
print("engine   :", L.config.engine_exe())
print("csm-data :", L.CSM_DATA, "@", L.CSM_DATA_COMMIT[:8])

engine   : C:\TEMP\dsn\dssat\dscsm048.exe
csm-data : C:\Users\benny\OneDrive\Documents\Github\dssat-csm-data @ 99c95b34


## 1. A sequence experiment file

A sequence file has the sections of any experiment file. What differs is the treatment table. In an ordinary
experiment each line is a treatment; in a sequence each line is one **component** of a rotation, and the
columns before the name say where it belongs:

- `N`: the sequence (treatment) the component belongs to;
- `R`: the rotation component number, the order in which the crops follow each other;
- `O`: rotation option and `C`: crop component (both 0 in the examples here).

The engine's reader shows the difference: in sequence mode `N` and `R` are read as two-digit numbers
([`ipexp.for:865-866`](https://github.com/DSSAT/dssat-csm-os/blob/25c93b9a2f73f8ec45bf441b198dac5fb7b22849/InputModule/ipexp.for#L865), formats 55 and 56). Only the first component has
initial conditions (`IC` level 1); the later ones have `IC` 0 and start from the soil the previous crop left.
A **fallow** is a component too, with crop code `FA`, ended by a harvest date. Every component has its own
simulation-control level (`SM`): its own planting rule, water and nitrogen switches, and harvest.

In [2]:
def section(text, name):
    i = text.index(name)
    return text[i:text.index("*", i + 1)].rstrip()

sq1 = (L.CSM_DATA / "Sequence/UFGA7803.SQX").read_text(encoding="latin-1")
sq2 = (L.CSM_DATA / "Sequence/ITHY7501.SQX").read_text(encoding="latin-1")
for t in (sq1, sq2):
    print(t.splitlines()[0]); print(section(t, "*TREATMENTS")); print(section(t, "*CULTIVARS")); print()
gen = [ln for ln in sq2.splitlines() if re.match(r"^ \d (GE|MA|PL|HA) ", ln)]
print("\n".join(gen))
L.export(pd.DataFrame([{"file": f, "section": s_, "text": section(t, s_)} for f, t in [("UFGA7803.SQX", sq1), ("ITHY7501.SQX", sq2)]
                       for s_ in ("*TREATMENTS", "*CULTIVARS")]), M, "sqx_sections")
L.export(pd.DataFrame({"line": gen}), M, "sqx_controls")

*EXP.DETAILS: UFGA7803SQ SEQUENCE ANALYSIS, SIMPLE MULTI-TREAT FILE, 5 YEARS; 30 REPS
*TREATMENTS                        -------------FACTOR LEVELS------------
@N R O C TNAME.................... CU FL SA IC MP MI MF MR MC MT ME MH SM
 1 1 1 0 Soybean treatment 1        1  1  0  1  1  1  0  0  0  0  0  0  1
 1 2 1 0 Fallow treatment 1         2  1  0  0  0  0  0  0  0  0  0  1  2
 2 1 1 0 Dry bean treatment 2       3  1  0  1  1  1  0  0  0  0  0  0  1
 2 2 1 0 Fallow treatment 2         2  1  0  0  0  0  0  0  0  0  0  1  2
*CULTIVARS
@C CR INGENO CNAME
 1 SB IB0001 BRAGG
 2 FA IB0001 -99
 3 BN IB0001 PORILLO SINTETICO

*EXP.DETAILS: ITHY7501SQ SOYBEAN-CHICKPEA ROTATION; 25 YEARS, 15 REPS
*TREATMENTS                        -------------FACTOR LEVELS------------
@N R O C TNAME.................... CU FL SA IC MP MI MF MR MC MT ME MH SM
 1 1 0 0 SOYBEAN                    1  1  0  1  1  0  1  0  0  0  0  0  1
 1 2 0 0 CHICKPEA                   2  1  0  0  2  0  0  0  0  0  0  0  2
 1 3 0

WindowsPath('C:/Users/benny/OneDrive/Documents/Github/xdssat/docs/learn/data/module6/sqx_controls.csv')

## 2. Running a sequence

The engine runs a sequence in mode `Q` from a batch file that lists the components in order: the file name in
the first 92 columns, then the treatment, replicate and sequence (component) numbers as `1X,I6` fields
([`CSM_Main/CSM.for`](https://github.com/DSSAT/dssat-csm-os/blob/25c93b9a2f73f8ec45bf441b198dac5fb7b22849/CSM_Main/CSM.for), the batch reader). It runs one line per crop season and starts again
from the first line for the next year, carrying the soil water, mineral and organic nitrogen, carbon and
residues from one crop to the next. `learnlib.run_sequence` writes the batch file and runs one sequence; two
sequences in one batch would be chained into each other, so each gets its own run.

`NYERS` sets the number of years and `NREPS` the number of replicates. Both example files generate their
weather (`WTHER` = `W`, from the climate files `UFGA.CLI` and `ITHY.CLI`), so each replicate is a different
draw of the same climate.

In [3]:
def seq_frame(s, labels):
    out = s[["RUNNO", "R#", "CR", "SDAT", "PDAT", "MDAT", "HWAM", "CWAM", "NFXM", "NUCM", "NLCM", "NIAM", "OCAM", "ONAM", "PRCM"]].copy()
    for c in out.columns.drop("CR"):
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out["replicate"] = (out.RUNNO == 1).cumsum()
    out["year"] = (out.SDAT // 1000).astype(int)
    out["crop"] = out.CR.map(labels)
    return out

t0 = time.time()
d = L.work_dir("m6_ufga")
L.stage(d, "Sequence/UFGA7803.SQX", "Soil/SOIL.SOL", "Weather/Climate/UFGA.CLI")
ufga = []
for trt, name in [(1, "Gainesville: soybean - fallow"), (2, "Gainesville: dry bean - fallow")]:
    s = L.run_sequence(d, "UFGA7803.SQX", trt, 2)
    ufga.append(seq_frame(s, {"SB": "soybean", "BN": "dry bean", "FA": "fallow"}).assign(sequence=name))
ufga = pd.concat(ufga)
print(f"UFGA7803: {len(ufga)} crop seasons in {time.time() - t0:.0f} s")
ufga.groupby(["sequence", "crop"])[["HWAM", "NFXM", "NLCM"]].mean().round(0)

UFGA7803: 660 crop seasons in 80 s


HWAM   NFXM  NLCM
sequence                       crop                         
Gainesville: dry bean - fallow dry bean  1128.0  103.0  13.0
                               fallow       0.0    0.0  40.0
Gainesville: soybean - fallow  fallow       0.0    0.0  33.0
                               soybean   3396.0  331.0  24.0

## 3. Soybean, chickpea and fallow at Hyderabad

`ITHY7501.SQX` grows soybean (PK-472) in the rainy season, chickpea (Annigeri) after it, and a fallow until the
next soybean, for 25 years with 15 weather replicates.

In [4]:
d = L.work_dir("m6_ithy")
L.stage(d, "Sequence/ITHY7501.SQX", "Soil/SOIL.SOL", "Weather/Climate/ITHY.CLI")
t0 = time.time()
ithy = seq_frame(L.run_sequence(d, "ITHY7501.SQX", 1, 3), {"SB": "soybean", "CH": "chickpea", "FA": "fallow"}).assign(sequence="Hyderabad: soybean - chickpea - fallow")
print(f"ITHY7501: {len(ithy)} crop seasons in {time.time() - t0:.0f} s")
ithy.groupby("crop")[["HWAM", "NFXM", "NLCM", "PRCM"]].median()

ITHY7501: 1125 crop seasons in 99 s


,HWAM,NFXM,NLCM,PRCM
crop,,,,
chickpea,339.0,16.0,0.0,39.0
fallow,0.0,0.0,0.0,156.0
soybean,3270.0,205.0,13.0,668.0


## 4. Creating a sequence experiment: the rotation without chickpea

To see what the chickpea does for the system, write a second sequence in the same file: soybean followed
directly by the fallow. In the treatment table that is two more lines, treatment 2, with the soybean and fallow
components of treatment 1 renumbered as rotation components 1 and 2. Nothing else changes: the fallow still ends
on its harvest date, before the next soybean.

In [5]:
trt_old = section(sq2, "*TREATMENTS")
lines = trt_old.splitlines()
soy = next(l for l in lines if l.startswith(" 1 1 "))
fal = next(l for l in lines if l.startswith(" 1 3 "))
new_trt = "\n".join(lines + [" 2 1" + soy[4:], " 2 2" + fal[4:]])
sq3 = L.replace_section(sq2, "TREATMENTS", new_trt)
sq3 = sq3.replace(sq2.splitlines()[0], "*EXP.DETAILS: ITHY7502SQ SOYBEAN WITH AND WITHOUT CHICKPEA, 25 YEARS")
print(section(sq3, "*TREATMENTS"))
d = L.work_dir("m6_ithy_nochk")
L.stage(d, "Soil/SOIL.SOL", "Weather/Climate/ITHY.CLI")
(d / "ITHY7502.SQX").write_text(sq3, encoding="latin-1")
t0 = time.time()
nochk = seq_frame(L.run_sequence(d, "ITHY7502.SQX", 2, 2), {"SB": "soybean", "CH": "chickpea", "FA": "fallow"}).assign(sequence="Hyderabad: soybean - fallow")
print(f"{len(nochk)} crop seasons in {time.time() - t0:.0f} s")
both = pd.concat([ithy, nochk])
both.groupby(["sequence", "crop"])[["HWAM", "NFXM", "NLCM", "NUCM"]].median()

*TREATMENTS                        -------------FACTOR LEVELS------------
@N R O C TNAME.................... CU FL SA IC MP MI MF MR MC MT ME MH SM
 1 1 0 0 SOYBEAN                    1  1  0  1  1  0  1  0  0  0  0  0  1
 1 2 0 0 CHICKPEA                   2  1  0  0  2  0  0  0  0  0  0  0  2
 1 3 0 0 FALLOW                     3  1  0  0  0  0  0  0  0  0  0  1  3
 2 1 0 0 SOYBEAN                    1  1  0  1  1  0  1  0  0  0  0  0  1
 2 2 0 0 FALLOW                     3  1  0  0  0  0  0  0  0  0  0  1  3


750 crop seasons in 72 s


HWAM   NFXM  NLCM   NUCM
sequence                               crop                                
Hyderabad: soybean - chickpea - fallow chickpea   339.0   16.0   0.0   25.0
                                       fallow       0.0    0.0   0.0    0.0
                                       soybean   3270.0  205.0  13.0   91.0
Hyderabad: soybean - fallow            fallow       0.0    0.0   0.0    0.0
                                       soybean   3317.0  190.0  28.0  112.0

In [6]:
def yearly(df):
    fal = df[df.crop == "fallow"].groupby(["sequence", "year"]).agg(OCAM_median=("OCAM", "median"), OCAM_p10=("OCAM", lambda x: x.quantile(0.1)),
                                                                 OCAM_p90=("OCAM", lambda x: x.quantile(0.9)), ONAM_median=("ONAM", "median"),
                                                                 NIAM_median=("NIAM", "median")).reset_index()
    crops = df[df.crop != "fallow"].groupby(["sequence", "crop", "year"]).HWAM.describe(percentiles=[0.1, 0.5, 0.9]).reset_index()
    crops = crops.rename(columns={"10%": "p10", "50%": "median", "90%": "p90"})[["sequence", "crop", "year", "p10", "median", "p90", "mean"]]
    return fal, crops

soil_y, crop_y = yearly(pd.concat([ithy, nochk, ufga]))
keep = ["sequence", "replicate", "year", "crop", "HWAM", "NFXM", "NUCM", "NLCM", "NIAM", "OCAM", "ONAM", "PRCM"]
L.export(pd.concat([ithy, nochk, ufga])[keep], M, "sequence_runs")
L.export(soil_y, M, "sequence_soil"); L.export(crop_y, M, "sequence_yields")
L.export(pd.DataFrame([{"section": "TREATMENTS", "text": section(sq3, "*TREATMENTS")}]), M, "sqx_new")
soil_y.groupby("sequence").apply(lambda g: pd.Series({"first": g.OCAM_median.iloc[0], "last": g.OCAM_median.iloc[-1]}))

,first,last
sequence,,
Gainesville: dry bean - fallow,38697.0,37310.5
Gainesville: soybean - fallow,38834.5,37847.0
Hyderabad: soybean - chickpea - fallow,91733.0,69048.0
Hyderabad: soybean - fallow,91535.0,62551.0


## 5. Things to try

- Give the chickpea irrigation at planting (an irrigation level for component 2) and compare the system.
- Remove the residue that the soybean leaves (a residue management level), and watch the soil carbon.
- Replace generated weather by measured Hyderabad weather (`WTHER` = `M`; dssat-csm-data has ITHY7501 to
  ITHY9901): does the ranking of the two sequences hold for the actual years?

In [7]:
info = pd.DataFrame([{"engine": str(L.config.engine_exe()), "csm_data_commit": L.CSM_DATA_COMMIT,
                      "csm_os_commit": L.CSM_OS_COMMIT, "run_on": time.strftime("%Y-%m-%d %H:%M")}])
L.export(info, M, "run_info")
info.T

,0
engine,C:\TEMP\dsn\dssat\dscsm048.exe
csm_data_commit,99c95b340acb9515c2d0a8ab582f5fbbf076f345
csm_os_commit,25c93b9a2f73f8ec45bf441b198dac5fb7b22849
run_on,2026-11-07 00:04
